# Test CCA

## Parametri

In [4]:
from pathlib import Path
from statsmodels.stats.multitest import multipletests
import sys
import joblib
import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from source.cca import load_tensor, pearson_corr

tensors_dir = project_root / "data" / "results" / "tensors"
results_dir = project_root / "data" / "results" / "cca"
models_dir = project_root / "data" / "model" 

arrangement_types = ["bass", "full", "full_no_bass", "harm", "lead"]

seed = 6
n_permutation = 5000

## Caricamento dei tensori e dei modelli

In [5]:
train = {}
test = {}
models = {}
for arrangement_type in arrangement_types:
    train[arrangement_type] = load_tensor(tensors_dir / f"tensor_train_{arrangement_type}.npz")
    test[arrangement_type] = load_tensor(tensors_dir / f"tensor_test_{arrangement_type}.npz")
    models[arrangement_type] = joblib.load(models_dir / f"cca_{arrangement_type}.joblib")


## Permutation test 

In [ ]:
def permutation_test(model, X, Y, n_permutation, seed):
    """permutation test per confermare la significatività della correlazione tra X e Y, usando il modello CCA già allenato"""
    rng = np.random.default_rng(seed)
    obs = pearson_corr(model, X, Y) # valore della correlazione per arrangiamento e batteria originali
    null = np.empty(n_permutation) # valore della correlazione se non c'è relazione tra arrangiamento e batteri
    for i in range(n_permutation):
        Yp = Y[rng.permutation(len(Y))] # batteria casuale
        null[i] = pearson_corr(model, X, Yp) # correlazione con il modello del train

    p_value = (1 + np.sum(null >= obs)) / (n_permutation + 1) 
    return obs, null, p_value

righe = []
for arrangement_type in arrangement_types:
    tensor_test = test[arrangement_type]
    obs, null, p_value = permutation_test(models[arrangement_type], tensor_test["X"], tensor_test["Y"], n_permutation, seed)
    righe.append(dict(classe=arrangement_type, 
                      n_train=len(train[arrangement_type]["ids"]), 
                      n_test=len(tensor_test["ids"]), 
                      observation=obs,
                      null_media=null.mean(), 
                      null_p95=np.percentile(null, 95), 
                      p_value=p_value,
                      n_permutation=n_permutation, 
                      seed=seed))

permutationTest = pd.DataFrame(righe)

# controllo della significatività dei p-value con correzione di Holm
reject, p_holm, _, _ = multipletests(permutationTest["p_value"], method="holm")
permutationTest["p_holm"] = p_holm
permutationTest["reject_holm"] = reject
permutationTest.to_parquet(results_dir / "permutation_test_conferma.parquet")